# 04 — Train A / D / B

Trains the three matched-encoder models on any of the five splits.

**Configuration cell below** selects the run set. Change `COMBOS` and re-run the training cell to run any subset of (model, split) combinations. Seed 42 for the current preliminary comparison; multi-seed reserved.

In [ ]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Import model definitions from the sibling scripts/models.py.
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from models import build_model, T_REF

RES = Path.cwd().parent / 'results'

## Hyperparameters and run set

In [ ]:
BATCH_SIZE   = 512
LR           = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS   = 100
PATIENCE     = 10
HIDDEN       = (256, 128)
DROPOUT      = 0.15
DEVICE       = 'cpu'
SEED         = 42

# Which (model, split) combinations to run in this notebook execution.
# Edit as needed.
COMBOS = [(m, s) for m in ('A', 'D', 'B')
                 for s in ('random', 'textrap', 'coldsol', 'coldpair', 'coldsolv')]
print('will run', len(COMBOS), 'combinations')

## Helpers

In [ ]:
def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed)

def load_features():
    data = np.load(RES / 'features.npz', allow_pickle=True)
    X_desc = np.concatenate([data['X_sol'], data['X_solv']], axis=1)
    return X_desc.astype(np.float32), data['T'].astype(np.float32), data['y'].astype(np.float32)

def load_split(name):
    d = np.load(RES / 'splits.npz')
    key = {
        'random':   ('random_train',   'random_val',   'random_test'),
        'textrap':  ('textrap_train',  'textrap_val',  'textrap_test'),
        'coldsol':  ('coldsol_train',  'coldsol_val',  'coldsol_test'),
        'coldpair': ('coldpair_train', 'coldpair_val', 'coldpair_test'),
        'coldsolv': ('coldsolv_train', 'coldsolv_val', 'coldsolv_test'),
    }[name]
    return d[key[0]], d[key[1]], d[key[2]]

def build_T_features(kind, T):
    if kind == 'A':
        return T.reshape(-1, 1).astype(np.float32)
    if kind == 'D':
        return np.stack([T, 1.0 / T], axis=1).astype(np.float32)
    if kind == 'B':
        return None
    raise ValueError(kind)

In [ ]:
def make_loader(x_desc, second, y, batch, shuffle):
    tensors = [torch.from_numpy(x_desc), torch.from_numpy(second), torch.from_numpy(y)]
    return DataLoader(TensorDataset(*tensors), batch_size=batch, shuffle=shuffle,
                      num_workers=0, pin_memory=False)

def evaluate(model, loader, kind, device):
    model.eval()
    preds, tgts = [], []
    with torch.no_grad():
        for batch in loader:
            if kind == 'B':
                x, Traw, y = [b.to(device) for b in batch]
                yhat = model(x, Traw)
            else:
                x, xT, y = [b.to(device) for b in batch]
                yhat = model(x, xT)
            preds.append(yhat.detach().cpu().numpy())
            tgts.append(y.detach().cpu().numpy())
    p = np.concatenate(preds); t = np.concatenate(tgts); err = p - t
    rmse = float(np.sqrt(np.mean(err**2)))
    mae  = float(np.mean(np.abs(err)))
    ss_res = float(np.sum(err**2))
    ss_tot = float(np.sum((t - t.mean())**2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    return dict(rmse=rmse, mae=mae, r2=r2, n=len(t))

In [ ]:
def train_one(kind, split_name, seed, device, verbose=True):
    set_seed(seed)
    X_desc, T, y = load_features()
    tr_idx, va_idx, te_idx = load_split(split_name)

    scaler_desc = StandardScaler().fit(X_desc[tr_idx])
    X_desc_scaled = scaler_desc.transform(X_desc).astype(np.float32)

    if kind in ('A', 'D'):
        T_feat = build_T_features(kind, T)
        scaler_T = StandardScaler().fit(T_feat[tr_idx])
        second_input = scaler_T.transform(T_feat).astype(np.float32)
    else:
        second_input = T.reshape(-1, 1).astype(np.float32)[:, 0]

    n_desc = X_desc_scaled.shape[1]
    model = build_model(kind, n_desc=n_desc, hidden_dims=HIDDEN, dropout=DROPOUT).to(device)
    if verbose:
        print(f'[{kind}/{split_name}] n_desc={n_desc}  params={sum(p.numel() for p in model.parameters()):,}')

    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()
    tr_loader = make_loader(X_desc_scaled[tr_idx], second_input[tr_idx], y[tr_idx], BATCH_SIZE, True)
    va_loader = make_loader(X_desc_scaled[va_idx], second_input[va_idx], y[va_idx], BATCH_SIZE, False)
    te_loader = make_loader(X_desc_scaled[te_idx], second_input[te_idx], y[te_idx], BATCH_SIZE, False)

    best_val = float('inf'); best_state = None; patience = 0
    t0 = time.time()
    for epoch in range(MAX_EPOCHS):
        model.train()
        for batch in tr_loader:
            if kind == 'B':
                x, Traw, yb = [b.to(device) for b in batch]; yhat = model(x, Traw)
            else:
                x, xT, yb = [b.to(device) for b in batch]; yhat = model(x, xT)
            loss = criterion(yhat, yb)
            optim.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
        v = evaluate(model, va_loader, kind, device)
        if v['rmse'] < best_val - 1e-4:
            best_val = v['rmse']
            best_state = {k: b.detach().cpu().clone() for k, b in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if verbose and (epoch % 10 == 0 or epoch == MAX_EPOCHS - 1):
            print(f'  epoch {epoch:3d}  val_rmse={v["rmse"]:.4f}  best={best_val:.4f}  patience={patience}')
        if patience >= PATIENCE:
            if verbose:
                print(f'  early stop at epoch {epoch}')
            break
    model.load_state_dict(best_state)
    elapsed = time.time() - t0
    metrics = {'train': evaluate(model, tr_loader, kind, device),
               'val':   evaluate(model, va_loader, kind, device),
               'test':  evaluate(model, te_loader, kind, device)}
    if verbose:
        print(f'  TRAIN rmse={metrics["train"]["rmse"]:.4f}  '
              f'VAL rmse={metrics["val"]["rmse"]:.4f}  '
              f'TEST rmse={metrics["test"]["rmse"]:.4f}  elapsed={elapsed:.1f}s')
    return metrics, elapsed

In [ ]:
def append_metrics(model_kind, split_name, seed, metrics, elapsed):
    row = {'model': model_kind, 'split': split_name, 'seed': seed,
           'elapsed_s': round(elapsed, 1)}
    for phase in ('train', 'val', 'test'):
        for k in ('rmse', 'mae', 'r2', 'n'):
            row[f'{phase}_{k}'] = metrics[phase][k]
    fn = RES / 'metrics.csv'
    df_row = pd.DataFrame([row])
    if fn.exists():
        old = pd.read_csv(fn)
        pd.concat([old, df_row], ignore_index=True).to_csv(fn, index=False)
    else:
        df_row.to_csv(fn, index=False)

## Run the selected combinations

In [ ]:
for k, s in COMBOS:
    print(f'\n===== {k} / {s} =====')
    metrics, elapsed = train_one(k, s, SEED, DEVICE)
    append_metrics(k, s, SEED, metrics, elapsed)

## Quick view of the metrics CSV

In [ ]:
df = pd.read_csv(RES / 'metrics.csv')
df[['model','split','seed','train_rmse','val_rmse','test_rmse','test_r2','elapsed_s']]